# 🤖 Chatbot using Hugging Face **Transformers**

 Step 1: Install Required Libraries

In [1]:
!pip install transformers torch -q

Step 2: Import Libraries

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print('✅ Libraries imported successfully.')
print(f'PyTorch version      : {torch.__version__}')

# Use GPU if available, otherwise CPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device being used    : {device.upper()}')

✅ Libraries imported successfully.
PyTorch version      : 2.10.0+cpu
Device being used    : CPU


Step 3: Load Pre-trained Transformer Model

We use **DialoGPT-medium** from Microsoft — a pre-trained transformer model specifically designed for **open-domain conversational AI**. It was trained on 147 million Reddit conversations and generates human-like dialogue responses.

| Model | Parameters | Best For |
|---|---|---|
| DialoGPT-small | 117M | Fast responses, basic conversation |
| DialoGPT-medium | 345M | Better quality, good balance |
| DialoGPT-large | 762M | Best quality, needs more RAM |

We use **medium** for a good balance of quality and speed.

In [3]:
MODEL_NAME = 'microsoft/DialoGPT-medium'

print(f'Loading model: {MODEL_NAME}')
print('This may take a minute on first run (downloading model weights)...\n')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side='left')
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model = model.to(device)
model.eval()

print('✅ Model and tokenizer loaded successfully.')
print(f'Model parameters     : ~345 million')
print(f'Model type           : GPT-2 based Causal LM')
print(f'Running on           : {device.upper()}')

Loading model: microsoft/DialoGPT-medium
This may take a minute on first run (downloading model weights)...



✅ Model and tokenizer loaded successfully.
Model parameters     : ~345 million
Model type           : GPT-2 based Causal LM
Running on           : CPU


Step 4: Build Response Generation Function

In [4]:
def generate_response(user_input, chat_history_ids=None, max_length=1000):
    new_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors='pt'
    ).to(device)

    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        bot_input_ids = new_input_ids

    if bot_input_ids.shape[-1] > max_length:
        bot_input_ids = bot_input_ids[:, -max_length:]

    with torch.no_grad():
        new_history_ids = model.generate(
            bot_input_ids,
            max_new_tokens   = 150,
            do_sample        = True,
            temperature      = 0.75,
            top_p            = 0.92,
            top_k            = 50,
            pad_token_id     = tokenizer.eos_token_id,
            repetition_penalty = 1.3
        )

    response = tokenizer.decode(
        new_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    ).strip()

    if not response:
        response = "I'm not sure I understand. Could you rephrase that?"

    return response, new_history_ids

print('✅ generate_response() function defined.')

✅ generate_response() function defined.


Step 5: Build the Interactive Chatbot

In [5]:
def run_chatbot():
    print('=' * 60)
    print('        🤖  AI CHATBOT ')
    print('=' * 60)
    chat_history_ids = None
    while True:
        try:
            user_input = input('You: ').strip()
        except (EOFError, KeyboardInterrupt):
            break
        if user_input.lower() in ['exit', 'quit']:
            break
        response, chat_history_ids = generate_response(user_input, chat_history_ids)
        print(f'Chatbot: {response}')

print('✅ run_chatbot() function defined.')

✅ run_chatbot() function defined.


Step 6: Start the Chatbot

In [7]:
run_chatbot()

Step 7: Demo — Simulated Conversation Output

In [9]:
def simulate_conversation(messages):
    print('=' * 60)
    print('  🤖  SIMULATED CONVERSATION DEMO')
    print('=' * 60)
    chat_history_ids = None
    for message in messages:
        print(f'You    : {message}')
        response, chat_history_ids = generate_response(message, chat_history_ids)
        print(f'Chatbot: {response}')

demo_messages = ["Hello!", "What is AI?"]
simulate_conversation(demo_messages)

Step 8: Understanding How the Model Works

In [10]:
print('📋 MODEL INFORMATION')
print('=' * 50)
print(f'Model Name     : microsoft/DialoGPT-medium')